# Quick Iteration Notebook for Exploring and Testing Data Files

### Author: CJJones

## Part 1: Raw State Legislative District Shape Files

In [1]:
# Package Imports
# !uv pip install geopandas
import os, re
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# Path/Directory Setup
ROOT_DIR = "./"
# DATA_DIR = os.path.join(ROOT_DIR)

UPPER_SHAPE_DIR = os.path.join(ROOT_DIR, "raw_sld_shapes/upper/")
LOWER_SHAPE_DIR = os.path.join(ROOT_DIR, "raw_sld_shapes/lower/")

UPPER_OUTPUT_PATH = "./data/us_sld_upper.gpkg"
LOWER_OUTPUT_PATH = "./data/us_sld_lower.gpkg"

# print(UPPER_SHAPE_DIR, "\n", LOWER_SHAPE_DIR, "\n")
print("Upper chamber files:", os.listdir(UPPER_SHAPE_DIR))
print("Lower chamber files:", os.listdir(LOWER_SHAPE_DIR))

Upper chamber files: ['tl_2024_54_sldu.zip', 'tl_2024_26_sldu.zip', 'tl_2024_21_sldu.zip', 'tl_2024_02_sldu.zip', 'tl_2024_46_sldu.zip', 'tl_2024_22_sldu.zip', 'tl_2024_04_sldu.zip', 'tl_2024_30_sldu.zip', 'tl_2024_55_sldu.zip', 'tl_2024_51_sldu.zip', 'tl_2024_27_sldu.zip', 'tl_2024_31_sldu.zip', 'tl_2024_39_sldu.zip', 'tl_2024_10_sldu.zip', 'tl_2024_15_sldu.zip', 'tl_2024_19_sldu.zip', 'tl_2024_08_sldu.zip', 'tl_2024_09_sldu.zip', 'tl_2024_11_sldu.zip', 'tl_2024_06_sldu.zip', 'tl_2024_35_sldu.zip', 'tl_2024_18_sldu.zip', 'tl_2024_24_sldu.zip', 'tl_2024_40_sldu.zip', 'tl_2024_47_sldu.zip', 'tl_2024_53_sldu.zip', 'tl_2024_41_sldu.zip', 'tl_2024_16_sldu.zip', 'tl_2024_32_sldu.zip', 'tl_2024_50_sldu.zip', 'tl_2024_25_sldu.zip', 'tl_2024_20_sldu.zip', 'tl_2024_23_sldu.zip', 'tl_2024_01_sldu.zip', 'tl_2024_45_sldu.zip', 'tl_2024_42_sldu.zip', 'tl_2024_28_sldu.zip', 'tl_2024_44_sldu.zip', 'tl_2024_38_sldu.zip', 'tl_2024_56_sldu.zip', 'tl_2024_33_sldu.zip', 'tl_2024_29_sldu.zip', 'tl_2024_13_

In [3]:
#  U.S. STATE AND TERRITORY FIPS CODE MAPPING
#  Source: ANSI INCITS 38-2009 / U.S. Census Bureau (TIGER)

FIPS_STATE_MAP = {
    "01": {"name": "Alabama", "abbr": "AL"},
    "02": {"name": "Alaska", "abbr": "AK"},
    "04": {"name": "Arizona", "abbr": "AZ"},
    "05": {"name": "Arkansas", "abbr": "AR"},
    "06": {"name": "California", "abbr": "CA"},
    "08": {"name": "Colorado", "abbr": "CO"},
    "09": {"name": "Connecticut", "abbr": "CT"},
    "10": {"name": "Delaware", "abbr": "DE"},
    "11": {"name": "District of Columbia", "abbr": "DC"},
    "12": {"name": "Florida", "abbr": "FL"},
    "13": {"name": "Georgia", "abbr": "GA"},
    "15": {"name": "Hawaii", "abbr": "HI"},
    "16": {"name": "Idaho", "abbr": "ID"},
    "17": {"name": "Illinois", "abbr": "IL"},
    "18": {"name": "Indiana", "abbr": "IN"},
    "19": {"name": "Iowa", "abbr": "IA"},
    "20": {"name": "Kansas", "abbr": "KS"},
    "21": {"name": "Kentucky", "abbr": "KY"},
    "22": {"name": "Louisiana", "abbr": "LA"},
    "23": {"name": "Maine", "abbr": "ME"},
    "24": {"name": "Maryland", "abbr": "MD"},
    "25": {"name": "Massachusetts", "abbr": "MA"},
    "26": {"name": "Michigan", "abbr": "MI"},
    "27": {"name": "Minnesota", "abbr": "MN"},
    "28": {"name": "Mississippi", "abbr": "MS"},
    "29": {"name": "Missouri", "abbr": "MO"},
    "30": {"name": "Montana", "abbr": "MT"},
    "31": {"name": "Nebraska", "abbr": "NE"},
    "32": {"name": "Nevada", "abbr": "NV"},
    "33": {"name": "New Hampshire", "abbr": "NH"},
    "34": {"name": "New Jersey", "abbr": "NJ"},
    "35": {"name": "New Mexico", "abbr": "NM"},
    "36": {"name": "New York", "abbr": "NY"},
    "37": {"name": "North Carolina", "abbr": "NC"},
    "38": {"name": "North Dakota", "abbr": "ND"},
    "39": {"name": "Ohio", "abbr": "OH"},
    "40": {"name": "Oklahoma", "abbr": "OK"},
    "41": {"name": "Oregon", "abbr": "OR"},
    "42": {"name": "Pennsylvania", "abbr": "PA"},
    "44": {"name": "Rhode Island", "abbr": "RI"},
    "45": {"name": "South Carolina", "abbr": "SC"},
    "46": {"name": "South Dakota", "abbr": "SD"},
    "47": {"name": "Tennessee", "abbr": "TN"},
    "48": {"name": "Texas", "abbr": "TX"},
    "49": {"name": "Utah", "abbr": "UT"},
    "50": {"name": "Vermont", "abbr": "VT"},
    "51": {"name": "Virginia", "abbr": "VA"},
    "53": {"name": "Washington", "abbr": "WA"},
    "54": {"name": "West Virginia", "abbr": "WV"},
    "55": {"name": "Wisconsin", "abbr": "WI"},
    "56": {"name": "Wyoming", "abbr": "WY"},
    "60": {"name": "American Samoa", "abbr": "AS"},
    "66": {"name": "Guam", "abbr": "GU"},
    "69": {"name": "Northern Mariana Islands", "abbr": "MP"},
    "72": {"name": "Puerto Rico", "abbr": "PR"},
    "74": {"name": "U.S. Minor Outlying Islands", "abbr": "UM"},
    "78": {"name": "U.S. Virgin Islands", "abbr": "VI"},
}

ex_idx = "08"
# Example usage:
print(FIPS_STATE_MAP[f"{ex_idx}"]["name"],
    ", ",
    FIPS_STATE_MAP[f"{ex_idx}"]["abbr"], 
    f" has the index number of {ex_idx}",
    sep="")


Colorado, CO has the index number of 08


In [4]:
upper_path = os.path.join(UPPER_SHAPE_DIR, "tl_2024_08_sldu.zip")
upper_gdf = gpd.read_file(f"zip://{upper_path}")

# Load lower (House) shapefile
lower_path = os.path.join(LOWER_SHAPE_DIR, "tl_2024_08_sldl.zip")
lower_gdf = gpd.read_file(f"zip://{lower_path}")

In [5]:
print("Upper Chamber Columns:")
print(upper_gdf.columns, "\n")

print("First few rows:")
display(upper_gdf.head(1))

# Optional: plot the shapes
# upper_gdf.plot(edgecolor="black", figsize=(8, 6))


Upper Chamber Columns:
Index(['STATEFP', 'SLDUST', 'GEOID', 'GEOIDFQ', 'NAMELSAD', 'LSAD', 'LSY',
       'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON',
       'geometry'],
      dtype='object') 

First few rows:


,STATEFP,SLDUST,GEOID,GEOIDFQ,NAMELSAD,LSAD,LSY,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,08,002,08002,610U900US08002,State Senate District 2,LU,2024,G5210,N,302288830,424206,+39.4517194,-104.7890653,"POLYGON ((-104.92163 39.32013, -104.9216 39.32..."


In [6]:
frames = []
zip_files = [f for f in os.listdir(UPPER_SHAPE_DIR) if f.endswith(".zip")]

for z in tqdm(zip_files, desc="Processing shapefiles"):
    path = os.path.join(UPPER_SHAPE_DIR, z)

    # Extract the 2-digit state FIPS code from the filename (tl_2024_XX_sldu.zip)
    match = re.search(r"tl_\d{4}_(\d{2})_sldu", z) # For Upper Directory
    # match = re.search(r"tl_\d{4}_(\d{2})_sldl", z) # For Lower Directory
    if not match:
        print(f"Skipping {z}: could not extract FIPS")
        continue
    fips = match.group(1)
    state_info = FIPS_STATE_MAP.get(fips, {"name": "Unknown", "abbr": "UNK"})

    gdf = gpd.read_file(f"zip://{path}")

    # Ensure consistent CRS (TIGER usually NAD83)
    gdf = gdf.to_crs("EPSG:4326")

    # Add helpful fields
    gdf["state_fips"] = fips
    gdf["state_name"] = state_info["name"]
    gdf["state_abbr"] = state_info["abbr"]

    frames.append(gdf)

# Combine into one GeoDataFrame
combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs="EPSG:4326")

print(f"Combined {len(frames)} states → {len(combined)} total districts")

os.makedirs(os.path.dirname(UPPER_OUTPUT_PATH), exist_ok=True)
combined.to_file(UPPER_OUTPUT_PATH, layer="sld_upper", driver="GPKG")

print(f"✅ Saved to {UPPER_OUTPUT_PATH}")

### LOWER SLD SECTION

frames = []
zip_files = [f for f in os.listdir(LOWER_SHAPE_DIR) if f.endswith(".zip")]

for z in tqdm(zip_files, desc="Processing shapefiles"):
    path = os.path.join(LOWER_SHAPE_DIR, z)

    # Extract the 2-digit state FIPS code from the filename (tl_2024_XX_sldu.zip)
    # match = re.search(r"tl_\d{4}_(\d{2})_sldu", z) # For Upper Directory
    match = re.search(r"tl_\d{4}_(\d{2})_sldl", z) # For Lower Directory
    if not match:
        print(f"Skipping {z}: could not extract FIPS")
        continue
    fips = match.group(1)
    state_info = FIPS_STATE_MAP.get(fips, {"name": "Unknown", "abbr": "UNK"})

    gdf = gpd.read_file(f"zip://{path}")

    # Ensure consistent CRS (TIGER usually NAD83)
    gdf = gdf.to_crs("EPSG:4326")

    # Add helpful fields
    gdf["state_fips"] = fips
    gdf["state_name"] = state_info["name"]
    gdf["state_abbr"] = state_info["abbr"]

    frames.append(gdf)

# Combine into one GeoDataFrame
combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs="EPSG:4326")

print(f"Combined {len(frames)} states → {len(combined)} total districts")

os.makedirs(os.path.dirname(LOWER_OUTPUT_PATH), exist_ok=True)
combined.to_file(LOWER_OUTPUT_PATH, layer="sld_upper", driver="GPKG")

print(f"✅ Saved to {LOWER_OUTPUT_PATH}")

Processing shapefiles: 100%|██████████| 52/52 [00:03<00:00, 15.39it/s]


Combined 52 states → 1964 total districts
✅ Saved to ./data/us_sld_upper.gpkg


Processing shapefiles: 100%|██████████| 49/49 [00:04<00:00, 11.66it/s]


Combined 49 states → 4779 total districts
✅ Saved to ./data/us_sld_lower.gpkg
